# Week 14 실습 — PBL(2) 베이스라인에서 한 걸음 더

**DNA 조절 영역 3-class 분류 (enhancer / promoter / open chromatin)**

지난 수업 Template 코드(`1-mer 문자빈도 + LogisticRegression`)에서 출발하여,
성능을 한 단계 끌어올리는 데 필요한 핵심 기법들을 한 노트북에서 직접 비교해봅니다.

**오늘의 흐름**
1. 데이터 불러오기 (지난 수업 그대로)
2. EDA — 클래스 분포 · 서열 길이 · GC content · N 비율
3. Representation 비교 — `1-mer` → `k-mer` → `TF-IDF` → `+handcrafted`
4. 공정한 평가(Fair evaluation) — `Stratified K-Fold CV` · `F1-macro` · `Confusion Matrix`
5. 다른 모델 비교 — `LogReg` · `RandomForest` · `GradientBoosting`
6. 다음 단계 — 1D CNN 미리보기

> 참고: 이 노트북은 **수업 시간 실습용**입니다. 코드를 직접 돌려보고, 출력 그래프와 표를
> 자기 팀 발표자료(중간/최종)에 그대로 가져다 쓸 수 있도록 설계되어 있습니다.

## 1. 환경 준비 및 데이터 불러오기

지난 수업 Template 코드와 동일한 방식으로 `dataset.csv` / `problem.csv` / `submission.csv` 를 다운로드합니다.
이미 받아두었다면 이 셀들은 그냥 통과합니다.

In [ ]:
project = 'dnasequence'  # 수정하지 마세요
username = ''            # 회원가입 시 사용한 이메일
password = ''            # 비밀번호

리더보드 제출을 위한 기본 설정: 아래 코드를 실행해주세요.

In [ ]:
import os
import urllib.request

if not os.path.exists('competition.py'):
    url = 'https://raw.githubusercontent.com/agtechresearch/LectureMLbasic/refs/heads/main/competition/competition.py'
    urllib.request.urlretrieve(url, 'competition.py')

# 데이터 다운로드: 미리 받아놓은 데이터가 없다면, 아래 코드를 실행하여 데이터를 다운로드하세요.
import competition
competition.download_competition_files(
    'https://raw.githubusercontent.com/agtechresearch/LectureMLbasic/main/dnasequence/bundle.zip',
    use_competition_url=False,
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 경고 무시
warnings.filterwarnings("ignore")

# Data 경로 설정
DATA_DIR = "data"

In [ ]:
# 학습에 사용할 과거 DNA 서열 data set 로드 (dataset.csv)
dataset = pd.read_csv(os.path.join(DATA_DIR, "dataset.csv"))

# problem set 로드 (problem.csv)
problemset = pd.read_csv(os.path.join(DATA_DIR, "problem.csv"))

In [ ]:
# 라벨 매핑: 데이터의 GC/k-mer 시그니처 기반으로 결정 (label 0 = GC 가장 높음, CpG island 시그니처)
LABEL_NAMES = {0: 'promoter', 1: 'enhancer', 2: 'open chromatin'}

print('학습 데이터:', dataset.shape, ' / 문제 데이터:', problemset.shape)

dataset.head()

## 2. EDA — 데이터를 먼저 본다

> **왜 EDA를 먼저?**  모델은 "본 적 없는 분포"를 잘 다루지 못합니다.
> 클래스가 얼마나 균형 잡혀 있는지, 서열의 길이는 어떻게 다른지, GC 함량은 어떤지를 먼저 보면
> *어떤 표현(representation)이 통할지* 가 보입니다.

### 2-1. 클래스 분포

In [ ]:
ax = dataset['label'].map(LABEL_NAMES).value_counts().plot(
    kind='bar', color=['#4A90E2', '#E05B4C', '#4CAF50'], rot=0,
    figsize=(6, 3.5))

ax.set_title('Class distribution (3-class)')
ax.set_ylabel('count')

for i, v in enumerate(dataset['label'].value_counts().sort_index()):
    ax.text(i, v + 200, str(v), ha='center', fontsize=10)
plt.tight_layout(); plt.show()

print(dataset['label'].value_counts(normalize=True).sort_index().round(3))

**해설.** 비율이 비슷한지(균형) / 한쪽으로 치우쳤는지(불균형)를 확인합니다.
불균형이면 `Accuracy` 만으로 평가하면 위험합니다 → **F1-macro** 가 더 안전합니다.

### 2-2. 서열 길이 분포

In [ ]:
dataset['len']    = dataset['seq'].str.len()
problemset['len'] = problemset['seq'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(dataset['len'], bins=40, color='#4A90E2', alpha=0.85)
axes[0].set_title('Sequence length (train)')
axes[0].set_xlabel('length'); axes[0].set_ylabel('count')

sns.violinplot(data=dataset, x='label', y='len', ax=axes[1],
               palette=['#4A90E2', '#E05B4C', '#4CAF50'])
axes[1].set_xticklabels([LABEL_NAMES[i] for i in range(3)])
axes[1].set_title('Length per class')
plt.tight_layout(); plt.show()

print(dataset.groupby('label')['len'].describe().round(1))

**해설.** 클래스별로 평균 길이가 다르다면 **길이 자체가 강한 단서**가 될 수 있습니다.
(예: enhancer 가 promoter 보다 평균적으로 더 짧다면 → 길이 한 개만으로도 분리력이 생김)

### 2-3. GC content (G+C 비율)

DNA의 GC 함량은 클래스마다 다른 경향을 보입니다.
예를 들어 promoter는 종종 CpG island 와 함께 GC 함량이 높게 나옵니다.

In [ ]:
def gc_ratio(s):
    s = s.upper()
    n_total = sum(c in 'ACGT' for c in s)  # N 은 분모에서 제외
    if n_total == 0: return 0.0
    return sum(c in 'GC' for c in s) / n_total

dataset['gc'] = dataset['seq'].apply(gc_ratio)

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.violinplot(data=dataset, x='label', y='gc', ax=ax,
               palette=['#4A90E2', '#E05B4C', '#4CAF50'])
ax.set_xticklabels([LABEL_NAMES[i] for i in range(3)])
ax.set_title('GC content per class'); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

print(dataset.groupby('label')['gc'].mean().round(3))

### 2-4. N 비율 (알 수 없는 염기)

`N` 은 약 5% 등장합니다. 클래스마다 결측이 한쪽으로 치우쳤다면 그것 자체가 신호일 수 있습니다.

In [ ]:
dataset['n_ratio'] = dataset['seq'].str.upper().apply(lambda s: s.count('N') / max(len(s), 1))

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.boxplot(data=dataset, x='label', y='n_ratio', ax=ax,
            palette=['#4A90E2', '#E05B4C', '#4CAF50'])
ax.set_xticklabels([LABEL_NAMES[i] for i in range(3)])
ax.set_title('N ratio per class'); ax.set_ylim(0, dataset['n_ratio'].quantile(0.99))
plt.tight_layout(); plt.show()

### 2-5. 클래스별 '특이 단어' — 3-mer 빈도 상위

글자 하나가 아니라 **3글자 묶음(3-mer)** 으로 보면 클래스 차이가 또렷해집니다.
각 클래스에서 다른 클래스보다 *상대적으로 더 자주 등장하는* 3-mer 들을 뽑아봅니다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 빠른 EDA용으로 클래스당 5,000개 샘플만 사용
rng = np.random.RandomState(0)
sub_idx = (dataset.groupby('label', group_keys=False)
                  .apply(lambda d: d.sample(min(5000, len(d)), random_state=0)).index)
sub = dataset.loc[sub_idx]

cv = CountVectorizer(analyzer='char', ngram_range=(3, 3), lowercase=False)
X3 = cv.fit_transform(sub['seq'])
vocab = np.array(cv.get_feature_names_out())

import pandas as pd
rows = []
for c in [0, 1, 2]:
    in_c   = sub['label'].values == c
    f_c    = np.asarray(X3[in_c].mean(axis=0)).ravel()
    f_rest = np.asarray(X3[~in_c].mean(axis=0)).ravel()
    diff   = f_c - f_rest
    top    = np.argsort(diff)[::-1][:8]
    rows.append(pd.Series(
        {f'{LABEL_NAMES[c]} top {i+1}': f'{vocab[idx]} ({diff[idx]:+.3f})'
         for i, idx in enumerate(top)}))
pd.DataFrame(rows, index=[LABEL_NAMES[i] for i in range(3)]).T

**해설.** 클래스별로 *튀어나오는 3-mer* 가 다르게 보입니다.
이게 곧 다음 절에서 다룰 **k-mer 표현이 통하는 이유** 입니다.

## 3. Representation 비교 — "글자" 에서 "단어" 로

동일한 모델(`LogisticRegression`)을 쓰고 **표현만 바꿔서** 성능 차이를 봅니다.
공정한 비교를 위해 학습/검증 분할은 한 번만 고정합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import hstack, csr_matrix

# 1) 학습용/검증용 인덱스를 한 번만 고정 (모든 표현이 같은 분할을 보도록)
Y = dataset['label'].values
idx_train, idx_val = train_test_split(
    np.arange(len(dataset)), test_size=0.2, stratify=Y, random_state=42)

seqs_train = dataset.iloc[idx_train]['seq'].values
seqs_val   = dataset.iloc[idx_val]['seq'].values
y_train, y_val = Y[idx_train], Y[idx_val]

print(f'train: {len(seqs_train):,} / val: {len(seqs_val):,}')

### 3-A. 1-mer (Template 베이스라인 재현)

In [ ]:
vec_1mer = CountVectorizer(analyzer='char', ngram_range=(1, 1), lowercase=False)
X_tr = vec_1mer.fit_transform(seqs_train)
X_va = vec_1mer.transform(seqs_val)

model = LogisticRegression(max_iter=2000, n_jobs=-1)
model.fit(X_tr, y_train)
y_pred = model.predict(X_va)

score_1mer = (accuracy_score(y_val, y_pred), f1_score(y_val, y_pred, average='macro'))
print(f'features: {X_tr.shape[1]}  ·  Val Acc: {score_1mer[0]:.4f}  ·  F1-macro: {score_1mer[1]:.4f}')
print(' vocab :', list(vec_1mer.get_feature_names_out()))

### 3-B. k-mer (3~5글자 묶음)

`ngram_range=(3, 5)` 로 3·4·5-mer 를 한꺼번에 셉니다.
차원이 커지므로 `min_df=5` 로 너무 드문 k-mer 는 자릅니다.

기존의 1-mer 보다 계산량이 폭발적으로 증가(k 증가에 따른 기하급수적 증가)하기 때문에 학습에도 많은 시간이 소요됩니다.

In [ ]:
vec_kmer = CountVectorizer(analyzer='char', ngram_range=(3, 5),
                            lowercase=False, min_df=5)
X_tr = vec_kmer.fit_transform(seqs_train)
X_va = vec_kmer.transform(seqs_val)

model = LogisticRegression(max_iter=2000, n_jobs=-1)
model.fit(X_tr, y_train)
y_pred = model.predict(X_va)

score_kmer = (accuracy_score(y_val, y_pred), f1_score(y_val, y_pred, average='macro'))
print(f'features: {X_tr.shape[1]:,}  ·  Val Acc: {score_kmer[0]:.4f}  ·  F1-macro: {score_kmer[1]:.4f}')

### 3-C. TF-IDF k-mer

`흔한 k-mer 는 약하게, 특정 클래스에서만 자주 나오는 k-mer 는 강하게` 가중합니다.

In [ ]:
vec_tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),
                             lowercase=False, min_df=5, sublinear_tf=True)
X_tr = vec_tfidf.fit_transform(seqs_train)
X_va = vec_tfidf.transform(seqs_val)

model = LogisticRegression(max_iter=2000, n_jobs=-1)
model.fit(X_tr, y_train)
y_pred = model.predict(X_va)

score_tfidf = (accuracy_score(y_val, y_pred), f1_score(y_val, y_pred, average='macro'))
print(f'features: {X_tr.shape[1]:,}  ·  Val Acc: {score_tfidf[0]:.4f}  ·  F1-macro: {score_tfidf[1]:.4f}')

### 3-D. TF-IDF k-mer  +  Handcrafted (GC, 길이, N비율)

사람이 "클래스를 가르는 데 도움이 될 것" 이라고 알고 있는 피처(GC content / 길이 / N비율)를 직접 만들어 옆에 붙입니다.

In [ ]:
def handcrafted_features(seqs):
    rows = []
    for s in seqs:
        s = s.upper()
        L = max(len(s), 1)
        n_atgc = sum(c in 'ACGT' for c in s) or 1
        gc = sum(c in 'GC' for c in s) / n_atgc
        n_ratio = s.count('N') / L
        rows.append([L, gc, n_ratio])
    return np.asarray(rows, dtype=float)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
H_tr   = scaler.fit_transform(handcrafted_features(seqs_train))
H_va   = scaler.transform(handcrafted_features(seqs_val))

# TF-IDF + handcrafted 를 옆으로 이어 붙이기
X_tr_full = hstack([vec_tfidf.transform(seqs_train), csr_matrix(H_tr)])
X_va_full = hstack([vec_tfidf.transform(seqs_val),   csr_matrix(H_va)])

model = LogisticRegression(max_iter=2000, n_jobs=-1)
model.fit(X_tr_full, y_train)
y_pred = model.predict(X_va_full)

score_full = (accuracy_score(y_val, y_pred), f1_score(y_val, y_pred, average='macro'))
print(f'features: {X_tr_full.shape[1]:,}  ·  Val Acc: {score_full[0]:.4f}  ·  F1-macro: {score_full[1]:.4f}')

### 3-E. 표현별 성능 정리

In [ ]:
results = pd.DataFrame({
    'Val Accuracy': [score_1mer[0], score_kmer[0], score_tfidf[0], score_full[0]],
    'F1-macro'    : [score_1mer[1], score_kmer[1], score_tfidf[1], score_full[1]],
}, index=['1-mer', 'k-mer(3-5)', 'TF-IDF(3-5)', 'TF-IDF + handcrafted'])
results = results.round(4)
results


In [ ]:
ax = results.plot(kind='barh', figsize=(8, 4),
                   color=['#4A90E2', '#E05B4C'])
ax.set_xlim(0.3, max(0.95, results.values.max() + 0.05))
ax.set_title('Representation 비교 — Val Acc & F1-macro')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout(); plt.show()

> **읽는 법.** 1-mer → k-mer 로 갈 때 가장 큰 점프가 일어납니다.
> TF-IDF 와 handcrafted 는 그 위에 "조금씩" 더 개선을 해줍니다.

## 4. 공정한 평가(fair evaluation) — CV · F1-macro · Confusion Matrix

한 번의 8:2 분할만 보면 *운이 좋아서 잘 나온 점수* 와 *실제 일반화 성능* 을 구분할 수 없습니다.
`Stratified K-Fold` 로 여러 번 나눠서 평균을 본 뒤, 그 모델로 **혼동 행렬**도 그립니다.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

# 최종 비교에서 가장 좋았던 표현으로 묶어서 다시 학습
best_pipe = make_pipeline(
    TfidfVectorizer(analyzer='char', ngram_range=(3, 5),
                     lowercase=False, min_df=5, sublinear_tf=True),
    LogisticRegression(max_iter=2000, n_jobs=-1),
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = cross_val_score(best_pipe, dataset['seq'].values, Y, cv=skf,
                          scoring='accuracy', n_jobs=-1)
cv_f1  = cross_val_score(best_pipe, dataset['seq'].values, Y, cv=skf,
                          scoring='f1_macro', n_jobs=-1)

print(f'5-Fold Accuracy : {cv_acc.mean():.4f}  ± {cv_acc.std():.4f}')
print(f'5-Fold F1-macro : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}')

### Confusion Matrix

어떤 클래스끼리 헷갈리는지 한눈에 보여줍니다. 발표자료에 그대로 가져다 쓰면 좋습니다.

In [ ]:
best_pipe.fit(seqs_train, y_train)
y_pred = best_pipe.predict(seqs_val)

cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[LABEL_NAMES[i] for i in range(3)],
            yticklabels=[LABEL_NAMES[i] for i in range(3)], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (TF-IDF k-mer + LogReg)')
plt.tight_layout(); plt.show()

print(classification_report(y_val, y_pred,
      target_names=[LABEL_NAMES[i] for i in range(3)], digits=3))

> **읽는 법.** 대각선이 진할수록 좋고, 대각선 밖이 두꺼우면 그 두 클래스를 **혼동** 한다는 뜻입니다.
> 예를 들어 enhancer ↔ open chromatin 이 자주 헷갈린다면, 두 클래스를 더 잘 가르는 피처가 필요한 단서입니다.

## 5. 모델 비교 — LogReg · RandomForest · GradientBoosting

같은 표현(`TF-IDF k-mer`) 위에서 모델만 바꿔서 비교합니다.
트리 계열은 희소 행렬에서 LogReg 보다 느릴 수 있으니, 메모리 절약을 위해
`min_df=10` 로 차원을 조금 줄여서 돌립니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

vec_cmp = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),
                           lowercase=False, min_df=10, sublinear_tf=True)
X_tr = vec_cmp.fit_transform(seqs_train)
X_va = vec_cmp.transform(seqs_val)
print('feature dim:', X_tr.shape[1])

models = {
    'LogReg'           : LogisticRegression(max_iter=2000, n_jobs=-1),
    'RandomForest'     : RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
    'GradientBoosting' : GradientBoostingClassifier(max_depth=3, n_estimators=150, random_state=0),
}

rows = []
for name, m in models.items():
    m.fit(X_tr, y_train)
    yp = m.predict(X_va)
    rows.append((name,
                 accuracy_score(y_val, yp),
                 f1_score(y_val, yp, average='macro')))
model_cmp = pd.DataFrame(rows, columns=['model', 'Val Acc', 'F1-macro']).set_index('model').round(4)
model_cmp

In [ ]:
ax = model_cmp.plot(kind='barh', figsize=(8, 3.5),
                     color=['#4A90E2', '#E05B4C'])
ax.set_xlim(0.3, max(0.95, model_cmp.values.max() + 0.05))
ax.set_title('모델 비교 — 같은 TF-IDF k-mer 표현 위에서')
for c in ax.containers:
    ax.bar_label(c, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout(); plt.show()

## 6. (선택) 리더보드 제출

지금까지 비교에서 가장 좋았던 조합을 골라 `problem.csv` 에 대해 예측한 뒤 제출합니다.

In [ ]:
# 가장 좋았던 파이프라인으로 전체 학습 데이터에 fit 후, 문제셋 예측
final_pipe = make_pipeline(
    TfidfVectorizer(analyzer='char', ngram_range=(3, 5),
                     lowercase=False, min_df=5, sublinear_tf=True),
    LogisticRegression(max_iter=2000, n_jobs=-1),
)
final_pipe.fit(dataset['seq'].values, Y)
problem_pred = final_pipe.predict(problemset['seq'].values)

submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
submission['label'] = problem_pred
submission.head()

In [ ]:
# 제출 (필요할 때만 주석 해제)
# competition.submit(project, username, password, submission)

## 7. 다음 단계 — 1D CNN 미리보기

k-mer 표현은 "몇 글자짜리 단어가 몇 번 등장했나" 만 봅니다.
**위치 정보**는 모두 버립니다.

다음주 실습에서는 위치까지 살리는 **1D CNN** 을 연습/실습해볼 예정입니다.
기본 아이디어는 이렇습니다:

1. DNA 서열을 `A/C/G/T/N` → **one-hot** (길이 × 5) 로 바꾼다
2. 짧은 `Conv1D` 필터 (예: width=6) 가 "모티프 검출기" 역할을 한다
3. `GlobalMaxPool1D` 로 "이 모티프가 어디에 있든 한 번이라도 강하게 매치되었는가" 를 본다
4. `Dense` 로 3-class 로 분류한다

이 흐름은 PBL(2) 평가 항목의
> *베이스라인 → 개선 모델로의 발전 과정* / *모델 비교 실험* / *Interpretability(모티프)*

에 그대로 매핑됩니다.

---

이 결과는 그대로 **중간발표(6/4)** 자료의 EDA · 베이스라인 · 1차 개선 · 평가 섹션에 사용할 수 있습니다.